In [307]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [308]:
import torch
import torch.nn as nn
import torch.functional as F
import math

from tokenizer import CharacterTokenizer

### Tokenization and Embeddings

Tokenization will be done based on a character-level tokenizer.

In [309]:
text = "Hello! I am learning how to build a transformer from scratch using PyTorch."

tokenizer = CharacterTokenizer(text)

tokens = tokenizer.encode(text)

print(tokenizer.decode(tokens) == text)


True


The dimensions of the embeddings will be
$$
E_{sum}=E_{token}+E_{position}
$$

The shape of these tensors are as follows
- $E_{sum}$ : $x_1 + x_2 \in \mathbb{R^{T \times {d_{embed}}}}$
- $E_{token}$ and $E_{position}$ : $x \in \mathbb{R^{T \times {d_{embed}}}}$

where $T$ is the size of the sequence of tokens


In [310]:
tokens = torch.tensor(tokenizer.encode(text), dtype=torch.long)
d_embed = 64

embedding_matrix = torch.randn(tokenizer.vocab_size, d_embed) / math.sqrt(d_embed)
pos_embedding_matrix = torch.randn(128, d_embed) / math.sqrt(d_embed)
positions = torch.arange(len(tokens))

embeddings = embedding_matrix[tokens]
embedded_tokens = pos_embedding_matrix[positions] + embeddings

print(pos_embedding_matrix.shape)
print(embedded_tokens.shape)
embedded_tokens

torch.Size([128, 64])
torch.Size([75, 64])


tensor([[-0.1613, -0.0317,  0.0632,  ...,  0.2866, -0.1129, -0.1295],
        [ 0.2476,  0.1087,  0.2335,  ..., -0.1831,  0.0256, -0.2409],
        [ 0.0018, -0.1992,  0.0060,  ...,  0.2490, -0.2608, -0.0430],
        ...,
        [-0.0986,  0.1345,  0.1870,  ...,  0.1481, -0.3072, -0.0760],
        [ 0.1724,  0.1897,  0.0367,  ...,  0.3077, -0.0556,  0.0166],
        [ 0.2346,  0.2757, -0.0494,  ...,  0.2538, -0.0523,  0.0915]])

### Self-Attention Mechanism

We compute the self-attention via computing 3 representations: $W_q, W_k$ and $W_v$



Then, we calculate the scaled dot product and apply a causal mask. The formula is:
$$
S=\frac{QK^T}{\sqrt{d_k}}
$$

After obtaining this result, we apply the softmax multiply by the value vector
$$
Attention(Q, K, V)=Softmax(\frac{QK^T}{\sqrt{d_k}})V
$$

In [311]:
w_q, w_k, w_v = (torch.randn(d_embed, d_embed) / math.sqrt(d_embed), 
                 torch.randn(d_embed, d_embed) / math.sqrt(d_embed), 
                 torch.randn(d_embed, d_embed) / math.sqrt(d_embed))

query_vector = torch.matmul(embedded_tokens, w_q)
key_vector = torch.matmul(embedded_tokens, w_k)
value_vector = torch.matmul(embedded_tokens, w_v)

scaled_dot_product = torch.matmul(query_vector, key_vector.T) / math.sqrt(d_embed)

triu_rows, triu_cols = torch.triu_indices(row=scaled_dot_product.size(0), 
                                  col=scaled_dot_product.size(1),
                                  offset=1)

scaled_dot_product[triu_rows, triu_cols] = float("-inf")

scaled_dot_product.shape
soft_max = nn.Softmax(dim=1)

out = soft_max(scaled_dot_product)
print(out.sum(dim=1) == torch.ones(out.shape[0], out.shape[1]))

A = torch.matmul(out, value_vector)
print(f"Output Shape: {A.shape} \n{A}")



tensor([[ True,  True, False,  ..., False,  True,  True],
        [ True,  True, False,  ..., False,  True,  True],
        [ True,  True, False,  ..., False,  True,  True],
        ...,
        [ True,  True, False,  ..., False,  True,  True],
        [ True,  True, False,  ..., False,  True,  True],
        [ True,  True, False,  ..., False,  True,  True]])
Output Shape: torch.Size([75, 64]) 
tensor([[ 0.2865, -0.1449, -0.0746,  ...,  0.3788, -0.1430, -0.2658],
        [ 0.3762, -0.1709, -0.1704,  ...,  0.3098, -0.1847, -0.1113],
        [ 0.2744, -0.0239, -0.0775,  ...,  0.2448, -0.0532, -0.0818],
        ...,
        [ 0.0350, -0.0069, -0.0074,  ...,  0.0219, -0.0178, -0.0720],
        [ 0.0326, -0.0089, -0.0098,  ...,  0.0248, -0.0204, -0.0686],
        [ 0.0329, -0.0055, -0.0077,  ...,  0.0241, -0.0183, -0.0608]])


### Multi-Head Attention

In [312]:
class MultiHeadAttention(nn.Module):
    def __init__(self, count_heads, d_embed):
        super().__init__()
        self.heads = count_heads
        self.d_embed = d_embed
        self.d_head = d_embed // count_heads

        self.w_q = nn.ParameterList(
            [nn.Parameter(torch.randn(d_embed, self.d_head) / math.sqrt(d_embed)) for _ in range(self.heads)])

        self.w_k = nn.ParameterList(
                [nn.Parameter(torch.randn(d_embed, self.d_head) / math.sqrt(d_embed)) for _ in range(self.heads)])

        self.w_v = nn.ParameterList(
                [nn.Parameter(torch.randn(d_embed, self.d_head) / math.sqrt(d_embed)) for _ in range(self.heads)])
        
        self.wO = nn.Parameter(torch.randn(d_embed, d_embed) / math.sqrt(d_embed))


    def forward(self, X):
        query_vector: list = [torch.matmul(X, w_q) for w_q in self.w_q]
        key_vector: list = [torch.matmul(X, w_k) for w_k in self.w_k]
        value_vector: list = [torch.matmul(X, w_v) for w_v in self.w_v]
        T: int = X.size(0)

        head_outputs = [self.__attention(query_vector[i], key_vector[i], value_vector[i], T) for i in range(self.heads)]
        attention_concat = torch.cat(head_outputs, dim=1)

        return torch.matmul(attention_concat, self.wO)


    def __attention(self, q_i, k_i, v_i, T):
        scaled_dot_product = torch.matmul(q_i, k_i.T) / math.sqrt(self.d_head)

        mask = torch.triu(
            torch.ones(T, T, dtype=torch.bool, device=scaled_dot_product.device),
            diagonal=1
        )
        masked_scores = scaled_dot_product.masked_fill(mask, float("-inf"))

        soft_max = nn.Softmax(dim=1)
        out = soft_max(masked_scores)

        attention_matrix = torch.matmul(out, v_i)

        return attention_matrix



In [313]:
device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print("Training device:", device)
embedded_tokens = embedded_tokens.to(device)

model = MultiHeadAttention(4, 64).to(device)
print(model)

out = model(embedded_tokens)
print(f"Output Shape: {out.shape}\n{out}")

MultiHeadAttention(
  (w_q): ParameterList(
      (0): Parameter containing: [torch.float32 of size 64x16]
      (1): Parameter containing: [torch.float32 of size 64x16]
      (2): Parameter containing: [torch.float32 of size 64x16]
      (3): Parameter containing: [torch.float32 of size 64x16]
  )
  (w_k): ParameterList(
      (0): Parameter containing: [torch.float32 of size 64x16]
      (1): Parameter containing: [torch.float32 of size 64x16]
      (2): Parameter containing: [torch.float32 of size 64x16]
      (3): Parameter containing: [torch.float32 of size 64x16]
  )
  (w_v): ParameterList(
      (0): Parameter containing: [torch.float32 of size 64x16]
      (1): Parameter containing: [torch.float32 of size 64x16]
      (2): Parameter containing: [torch.float32 of size 64x16]
      (3): Parameter containing: [torch.float32 of size 64x16]
  )
)
Output Shape: torch.Size([75, 64])
tensor([[-0.0691,  0.1543,  0.3148,  ...,  0.1112, -0.1566,  0.1076],
        [ 0.1047,  0.1235,  0.186

### Feed-Forward and Layer Normalization

In [314]:
class FeedForward(nn.Module):
    def __init__(self, d_embed):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(d_embed, 256),
            nn.ReLU(),
            nn.Linear(256, d_embed)
        )
        for layer in self.fc:
            if isinstance(layer, nn.Linear):
                nn.init.normal_(layer.weight, mean=0.0, std=1 / math.sqrt(layer.in_features))

    def forward(self, X):
        out = self.fc(X)
        return out


class LayerNorm(nn.Module):
    def __init__(self, d_embed, eps=1e-5):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_embed))
        self.beta = nn.Parameter(torch.zeros(d_embed))
        self.eps = eps

    def forward(self,x):
        mean = torch.mean(x, dim=1, keepdim=True)
        variance = torch.var(x, dim=1, keepdim=True, correction=0)

        return ((x - mean) / torch.sqrt(variance + self.eps)) * self.gamma + self.beta



### Transformer 

In [315]:
class TransformerBlock(nn.Module):
    def __init__(self, num_heads, d_embed):
        super().__init__()
        self.attention = MultiHeadAttention(num_heads, d_embed)
        self.ffn = FeedForward(d_embed)
        
        self.norm1 = LayerNorm(d_embed)
        self.norm2 = LayerNorm(d_embed)

    def forward(self, X):
        x1 = X + self.attention(self.norm1(X))
        x2 = x1 + self.ffn(self.norm2(x1))

        return x2


In [316]:
transform = TransformerBlock(4, 64).to(device)
output = transform(embedded_tokens)

print(f"Embedded Tokens Shape: {embedded_tokens.shape}\nTransform Output Shape: {output.shape}")

Embedded Tokens Shape: torch.Size([75, 64])
Transform Output Shape: torch.Size([75, 64])


In [317]:
class AutoRegressiveTransformer(nn.Module):
    def __init__(self, vocab_size, d_embed, num_heads, max_context_length, num_blocks=3):
        super().__init__()
        self.vocab_size = vocab_size
        self.num_heads = num_heads
        self.d_embed = d_embed
        self.d_head = d_embed // num_heads

        self.w_vocab = nn.Parameter(
            torch.randn(d_embed, vocab_size) / math.sqrt(d_embed))

        self.bias = nn.Parameter(
                torch.zeros(vocab_size))

        self.token_embeddings = nn.Parameter(
            torch.randn(vocab_size, d_embed) / math.sqrt(d_embed))
        
        self.pos_embeddings = nn.Parameter(
            torch.randn(max_context_length, d_embed) / math.sqrt(d_embed))

        self.layers = nn.ModuleList(
            [TransformerBlock(self.num_heads, self.d_embed) for _ in range(num_blocks)])

        self.transformer_sequence = nn.Sequential(*self.layers)
        self.layer_norm_final = LayerNorm(d_embed)


    def forward(self, tokens):
        embedded_tokens = self.token_embeddings[tokens] + self.pos_embeddings[:tokens.size(-1)]

        output = self.transformer_sequence(embedded_tokens)
        output = self.layer_norm_final(output)

        logits = torch.matmul(output, self.w_vocab) + self.bias

        return logits;


In [318]:
def create_sequences(tokens, seq_length, start_index=0):
    return (
        tokens[start_index:start_index + seq_length].to(device),
        tokens[start_index + 1:start_index + seq_length + 1].to(device)
    )

### Example Training

In [319]:
text = "the cat sat on the mat. the cat ate a fish."

tokens = torch.tensor(tokenizer.encode(text))
X, Y = create_sequences(tokens, seq_length = len(tokens) - 1)

vocab_size = tokenizer.vocab_size
d_embed = 64
num_heads = 4
max_context_length = 64
num_blocks = 3
batch_size = 1 # number of sequences

model = AutoRegressiveTransformer(
    vocab_size, d_embed, num_heads, max_context_length, num_blocks).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
criterion = nn.CrossEntropyLoss()

In [320]:
def run_training_epoch(X, Y):
    model.train()
    logits = model(X)

    loss = criterion(logits, Y)
    optimizer.zero_grad()

    loss.backward()
    optimizer.step()

    return loss.item()

In [324]:

import time

def generate_text(prompt: str, max_new_tokens=50):
    tokens = torch.tensor(
        tokenizer.encode(prompt),
        dtype=torch.long,
        device=next(model.parameters()).device,
    )

    model.eval()
    with torch.no_grad():
        print(f"Prompt: {prompt}")
        print(f"Prediction: {prompt}", end="", flush=True)

        time.sleep(1)
        for _ in range(max_new_tokens):

            context = tokens[-max_context_length:]
            logits = model(context)
      
            next_token = torch.argmax(logits[-1])

            # unsqueeze is used to rehsape () to (1, )
            tokens = torch.cat((tokens, next_token.unsqueeze(0)))
            
            print(tokenizer.decode([int(next_token.item())]), end="", flush=True)
            time.sleep(0.05)
    

In [325]:

generate_text("the cat", max_new_tokens=80)

Prompt: the cat
Prediction: the cat sat on the mat. the cat ate a fish. athe ath. ath. at the aat.at atat aatat at 

### Training on a larger text corpus

In [327]:
sentences = sample_corpus.splitlines()

train_text = "\n".join(sentences[:90])
val_text = "\n".join(sentences[90:])

tokenizer = CharacterTokenizer(train_text)
train_tokens = torch.tensor(tokenizer.encode(train_text), dtype=torch.long)
val_tokens = torch.tensor(tokenizer.encode(val_text), dtype=torch.long)



Update `create_sequences()` to use random start indexes

In [328]:
def create_sequences(tokens, seq_length):
    start_index = torch.randint(
        0, len(tokens) - seq_length, (1,)
    ).item()

    X = tokens[start_index:start_index + seq_length]
    Y = tokens[start_index + 1:start_index + seq_length + 1]

    return X.to(device), Y.to(device)

In [329]:
def run_train_step(X, Y):
    model.train()
    logits = model(X)

    loss = criterion(logits, Y)
    optimizer.zero_grad()

    loss.backward()
    optimizer.step()

    return loss.item()

Reinitialize the model

In [330]:
vocab_size = tokenizer.vocab_size

model = AutoRegressiveTransformer(
    vocab_size, d_embed, num_heads, max_context_length, num_blocks
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=3e-4, weight_decay=1e-2
)

In [ ]:
losses = []
num_steps = 10000
context_length = 64

train_tokens = torch.tensor(tokenizer.encode(train_text), dtype=torch.long)

for step in range(num_steps):
    X, Y = create_sequences(train_tokens, context_length)

    loss = run_training_epoch(X, Y)
    losses.append(loss)

    if step % 100 == 0:
        print(f"Step {step}: Loss = {loss:.4f}")

In [ ]:
generate_text("the cat and the kitten", max_new_tokens=30)